# 02 – Features : transformer les mesures en indicateurs de dégradation

**Objectif :** montrer ce que fait `src/features.py` et **pourquoi** ces variables aident à prédire une panne. Ce notebook n'écrit aucun fichier de données : il utilise les fonctions de `src/features.py` pour illustrer et vérifier.

**Idée :** une mesure isolée (une température de 68 °C) dit peu de choses. Ce qui compte, c'est l'**évolution** : la température monte-t-elle ? Les vibrations sont-elles plus fortes qu'au début de vie ?

**Prérequis :** avoir lancé `python src/generate_data.py` puis `python src/features.py` (ils créent `data/synthetic/` et `data/processed/features_temporal.csv`).

Deux familles de features :
1. **Temporelles** (sur les mesures de capteurs, toutes les 6 h) → `temporal_features`
2. **Vibratoires** (sur le signal brut, 1 fenêtre de 0,5 s) → `signal_features`

## 1. Chargement

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "features.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from features import temporal_features, signal_features, SENSOR_COLS
(ROOT / "reports/figures").mkdir(parents=True, exist_ok=True)

for f in ["data/synthetic/sensors.csv", "data/synthetic/signals.npy", "data/processed/features_temporal.csv"]:
    if not (ROOT / f).exists():
        raise FileNotFoundError(f"{f} introuvable : lance d'abord `python src/generate_data.py` puis `python src/features.py`.")

sensors = pd.read_csv(ROOT / "data/synthetic/sensors.csv")
print(sensors.shape, "|", sensors.machine_id.nunique(), "machines |", sensors.groupby(["machine_id", "cycle_id"]).ngroups, "cycles")
sensors.head(3)

## 2. Les features temporelles

`temporal_features` calcule, **pour chaque capteur**, machine par machine **et cycle par cycle** (jamais de mélange entre deux cycles de vie) :

| Feature | Sens |
|---|---|
| `capteur_mean_4` | moyenne des 4 dernières mesures (= 24 h) : le niveau récent, sans le bruit |
| `capteur_std_4` | écart-type sur 24 h : la machine est-elle instable ? |
| `capteur_slope_12` | pente sur 12 mesures (= 72 h) : le capteur monte ou descend ? |
| `capteur_ratio_baseline` | mesure ÷ moyenne des 10 premières mesures du cycle : écart à l'état « neuf » |

Capteurs : `vib_rms`, `vib_kurtosis`, `vib_crest`, `temperature`, `pressure`, `current` → 6 capteurs × 4 = **24 features** (+ les 6 mesures brutes = 30 variables utilisées par les modèles).

#### Exemple sur un cycle qui se termine par une panne

In [ ]:
# on prend un cycle terminé par une panne (rul_h connue)
cyc = sensors.dropna(subset=["rul_h"]).groupby(["machine_id", "cycle_id"]).size()
mid, cid = cyc[cyc > 150].index[0]
brut = sensors[(sensors.machine_id == mid) & (sensors.cycle_id == cid)]
ex = temporal_features(brut)
print(f"Machine {mid}, cycle {cid} : {len(ex)} mesures, panne à {ex.age_h.max() + 6:.0f} h")

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
axes[0].plot(ex.age_h, ex.vib_rms, alpha=.4, label="mesure brute")
axes[0].plot(ex.age_h, ex.vib_rms_mean_4, label="moyenne 4 mesures")
axes[0].set_ylabel("vib_rms (mm/s)"); axes[0].legend()
axes[1].plot(ex.age_h, ex.vib_rms_std_4, color="tab:orange"); axes[1].set_ylabel("écart-type (24 h)")
axes[2].plot(ex.age_h, ex.vib_rms_slope_12, color="tab:green"); axes[2].axhline(0, color="gray", lw=.8)
axes[2].set_ylabel("pente (72 h)")
axes[3].plot(ex.age_h, ex.vib_rms_ratio_baseline, color="tab:red"); axes[3].axhline(1, color="gray", lw=.8)
axes[3].set_ylabel("ratio / baseline"); axes[3].set_xlabel("âge dans le cycle (h)")
for ax in axes:
    ax.axvspan(ex.age_h.max() - 48, ex.age_h.max(), color="red", alpha=.12)   # zone « panne dans 48 h »
axes[0].set_title(f"Features de vib_rms, machine {mid} (zone rouge = 48 h avant la panne)")
plt.tight_layout()
plt.savefig(ROOT / "reports/figures/features_exemple_cycle.png", dpi=150)
plt.show()

**Lecture :** la mesure brute est bruitée ; la moyenne la lisse. Le **ratio à la baseline** passe de ~1 (état neuf) à une valeur nettement plus grande en fin de vie : c'est l'indicateur de dégradation le plus lisible. La zone rouge est ce que le classifieur doit apprendre à reconnaître.

## 3. Vérification : aucune fuite du futur

Une feature à l'instant *t* ne doit dépendre que des mesures **jusqu'à *t***. Sinon le modèle « voit l'avenir » et ses scores sont faux. Test : on **modifie fortement le futur** d'un cycle et on vérifie que les features du passé ne changent pas.

In [ ]:
t0 = brut.time_h.iloc[len(brut) // 2]            # milieu du cycle
modif = brut.copy()
modif.loc[modif.time_h >= t0, SENSOR_COLS] *= 10   # on bouleverse tout ce qui est après t0

a = temporal_features(brut)
b = temporal_features(modif)
cols = [c for c in a.columns if any(c.startswith(s) for s in SENSOR_COLS)]
passe = a.time_h < t0
print("Features du passé inchangées :", np.allclose(a.loc[passe, cols], b.loc[passe, cols], equal_nan=True))

## 4. Les features vibratoires

Le capteur de vibration fournit aussi le **signal brut** (1024 points à 2048 Hz = 0,5 s). `signal_features` en extrait :
- **RMS, peak, crest factor** : l'énergie et la présence de chocs ;
- **kurtosis** : mesure la présence de chocs ; elle augmente quand un défaut de roulement apparaît ;
- **énergie par bande de fréquence** (FFT) : une machine qui se dégrade répartit son énergie autrement entre les bandes ;
- **fréquence dominante, centroïde spectral**.

Comparons une fenêtre **saine** (début de cycle) et une fenêtre **dégradée** (juste avant la panne) de la même machine.

In [ ]:
signals = np.load(ROOT / "data/synthetic/signals.npy")
idx = pd.read_csv(ROOT / "data/synthetic/signals_index.csv")

# un cycle terminé par une panne, avec assez de fenêtres
c = idx.dropna(subset=["rul_h"]).groupby(["machine_id", "cycle_id"]).filter(lambda g: len(g) > 10)
g = c[(c.machine_id == c.machine_id.iloc[0]) & (c.cycle_id == c.cycle_id.iloc[0])]
sain, degrade = g.loc[g.age_h.idxmin()], g.loc[g.rul_h.idxmin()]
fs = sain.fs_hz

fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for j, (nom, r) in enumerate([("saine", sain), ("dégradée", degrade)]):
    x = signals[int(r.signal_idx)]
    axes[0, j].plot(np.arange(300) / fs * 1000, x[:300], lw=.8)
    axes[0, j].set_title(f"Fenêtre {nom} (RUL = {r.rul_h:.0f} h)"); axes[0, j].set_xlabel("temps (ms)")
    spec = np.abs(np.fft.rfft((x - x.mean()) * np.hanning(len(x)))) / len(x)
    axes[1, j].plot(np.fft.rfftfreq(len(x), 1 / fs), spec, lw=.8); axes[1, j].set_xlim(0, 1000)
    axes[1, j].set_xlabel("fréquence (Hz)"); axes[1, j].set_ylabel("amplitude")
plt.tight_layout()
plt.savefig(ROOT / "reports/figures/features_signal_sain_vs_degrade.png", dpi=150)
plt.show()

comp = pd.DataFrame({"saine": signal_features(signals[int(sain.signal_idx)], fs),
                     "dégradée": signal_features(signals[int(degrade.signal_idx)], fs)})
comp.round(3)

**Lecture :** sur la fenêtre dégradée, le signal contient des chocs périodiques, le **kurtosis** et le **crest factor** augmentent, et le spectre montre plus d'énergie dans les bandes 60–200 Hz et 500–1000 Hz (la fréquence dominante passe de 24 Hz à 48 Hz). Ces indicateurs décrivent la *forme* du signal, en plus de son énergie globale (RMS).

## 5. Quelles features sont liées à la panne imminente ?

Corrélation de chaque feature avec la cible `failure_within_h` (panne dans les 48 h), sur toutes les mesures.

In [ ]:
feat = pd.read_csv(ROOT / "data/processed/features_temporal.csv")
cols = [c for c in feat.columns if any(c.startswith(s) for s in SENSOR_COLS)]
corr = feat[cols].corrwith(feat["failure_within_h"]).sort_values()
top = pd.concat([corr.head(5), corr.tail(10)]).drop_duplicates().sort_values()
top.plot(kind="barh", figsize=(7, 5), color=np.where(top > 0, "tomato", "steelblue"))
plt.title("Corrélation avec « panne dans les 48 h »")
plt.xlabel("corrélation")
plt.tight_layout()
plt.savefig(ROOT / "reports/figures/features_correlation_cible.png", dpi=150)
plt.show()

## À retenir

- Les features **temporelles** résument l'évolution récente de chaque capteur ; le **ratio à la baseline** est le plus parlant.
- Les features **vibratoires** (kurtosis, crest factor, bandes FFT) détectent un défaut de roulement avant qu'il soit visible dans le RMS.
- Tout est calculé **par machine et par cycle, avec le passé seulement** (vérifié au §3).
- Les corrélations sont une première intuition ; l'importance réelle sera mesurée avec les modèles (`03b`) puis **SHAP**.
- Les données sont **synthétiques** : la dégradation est simulée, plus propre qu'en vrai.